# Frequency Branch Perception Experiment: DeepDetect
Standalone notebook testing three improvements to frequency branch perception.
No changes to the codebase - everything lives here.
The best approach will be transferred after evaluation.

Baseline to beat: **71.2% test accuracy** (v5 skin-tone patch, no augmentations, 30 epochs)

| ID | Approach | Description |
|---|---|---|
| P1 | Phase channel | Add FFT phase as 4th input channel alongside log-magnitude |
| P2 | Multi-scale FFT | Three patch sizes (28, 56, 112), shared CNN, concatenated features |
| P3 | MobileNetV3 backbone | Replace FrequencyCNN with pretrained MobileNetV3-Small |

All experiments use v5 patch selector, no augmentations, 30 epochs.


## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
import importlib
import numpy as np

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


Device: cuda
GPU:    NVIDIA RTX PRO 4000 Blackwell
VRAM:   25.2 GB


## 2. Data — v5 patch, no augmentations

In [2]:
from config import Config
from data.deepdetect import get_deepdetect_loaders

def make_cfg(experiment_name, epochs=30):
    cfg = Config()
    cfg.data.deepdetect_root   = "../data/raw/deep_detect/data"
    cfg.data.dataset           = "deepdetect"
    cfg.data.image_size        = 224
    cfg.data.batch_size        = 64
    cfg.data.num_workers       = 4
    cfg.backbone.img_size      = 224
    cfg.frequency.patch_size   = 56
    cfg.data.jpeg_aug          = False
    cfg.data.blur_aug          = False
    cfg.data.noise_aug         = False
    cfg.data.recompression_aug = False
    cfg.data.mixed_aug         = False
    cfg.train.epochs           = epochs
    cfg.train.backbone_lr      = cfg.train.lr
    cfg.experiment_name        = experiment_name
    cfg.notes                  = f"perception experiment: {experiment_name}"
    return cfg

cfg_data = make_cfg("perc_setup")
train_loader, val_loader, test_loader = get_deepdetect_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## 3. Shared Utilities

In [ ]:
# ── v5 patch selector (from patch experiment) ─────────────────────────────────
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denorm(t):
    return (t * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)

def rgb_to_hsv(rgb):
    r, g, b = rgb[0], rgb[1], rgb[2]
    maxc = rgb.max(0).values
    minc = rgb.min(0).values
    v    = maxc
    s    = torch.where(maxc != 0, (maxc - minc) / (maxc + 1e-8),
                       torch.zeros_like(maxc))
    rc   = (maxc - r) / (maxc - minc + 1e-8)
    gc   = (maxc - g) / (maxc - minc + 1e-8)
    bc   = (maxc - b) / (maxc - minc + 1e-8)
    h    = torch.where(r == maxc, bc - gc,
           torch.where(g == maxc, 2.0 + rc - bc, 4.0 + gc - rc))
    h    = (h / 6.0) % 1.0
    return h, s, v

def select_patch_v5(image, patch_size=56, stride=8):
    C, H, W = image.shape
    patch_size = min(patch_size, H, W)
    if patch_size == H and patch_size == W:
        return image
    img_denorm = denorm(image.cpu())
    h, s, v    = rgb_to_hsv(img_denorm)
    mask       = ((h >= 0.0) & (h <= 0.1) & (s >= 0.1) &
                  (s <= 0.7) & (v >= 0.2)).float().unsqueeze(0).unsqueeze(0)
    gray   = image.mean(dim=0, keepdim=True).unsqueeze(0)
    kernel = torch.ones(1, 1, patch_size, patch_size,
                        device=image.device) / (patch_size ** 2)
    local_mean    = F.conv2d(gray,      kernel, padding=0)
    local_mean_sq = F.conv2d(gray ** 2, kernel, padding=0)
    local_var     = (local_mean_sq - local_mean ** 2).squeeze()
    skin_kernel   = torch.ones(1, 1, patch_size, patch_size) / (patch_size ** 2)
    local_skin    = F.conv2d(mask, skin_kernel, padding=0).squeeze()
    skin_present  = (local_skin >= 0.3)
    if skin_present.any():
        local_var_masked = local_var.clone()
        local_var_masked[~skin_present] = float("inf")
        flat_idx = local_var_masked.argmin()
    else:
        flat_idx = local_var.argmin()
    top  = flat_idx // local_var.shape[1]
    left = flat_idx  % local_var.shape[1]
    return image[:, top:top+patch_size, left:left+patch_size]

def select_patch_v5_batch(images, patch_size=56):
    return torch.stack([select_patch_v5(images[i], patch_size)
                        for i in range(images.shape[0])])

from utils.metrics import binary_accuracy
from pathlib import Path

def train_and_eval(model, cfg, train_loader, val_loader, test_loader, device):
    """Generic training loop with train_acc reporting. Returns test accuracy."""
    epochs    = cfg.train.epochs
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler(enabled=device.type == "cuda")
    Path(cfg.train.checkpoint_dir).mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"Experiment: {cfg.experiment_name} | Epochs: {epochs}")
    print(f"{'='*65}")

    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        total_loss   = 0.0
        train_logits = []
        train_labels = []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(images)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            train_logits.append(logits.detach().cpu())
            train_labels.append(labels.cpu())

        scheduler.step()
        train_acc = binary_accuracy(torch.cat(train_logits), torch.cat(train_labels))

        model.eval()
        val_logits, val_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                val_logits.append(model(images.to(device)).cpu())
                val_labels.append(labels)
        val_acc = binary_accuracy(torch.cat(val_logits), torch.cat(val_labels))

        print(f"Epoch {epoch+1:>3}/{epochs} | "
              f"train_loss={total_loss/len(train_loader):.4f} | "
              f"train_acc={train_acc:.1%} | val_acc={val_acc:.1%}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(),
                       f"{cfg.train.checkpoint_dir}/best_{cfg.experiment_name}.pt")
            print(f"  -> Saved best val_acc={best_val_acc:.1%}")

    # Final test evaluation
    model.load_state_dict(torch.load(
        f"{cfg.train.checkpoint_dir}/best_{cfg.experiment_name}.pt",
        map_location=device))
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            all_logits.append(model(images.to(device)).cpu())
            all_labels.append(labels)
    acc = binary_accuracy(torch.cat(all_logits), torch.cat(all_labels))
    print(f"\nTest accuracy: {acc:.1%}")
    return acc

## 4. P1: Phase Channel
Add FFT phase as a 4th channel alongside log-magnitude.
Phase carries complementary information about the spatial arrangement of
frequency components. Generation artifacts often manifest in both magnitude
and phase.


In [ ]:
from models.frequency_branch import SRMFilterLayer

class PhaseAugmentedFFT(nn.Module):
    """
    Extended FFT representation: log-magnitude (3ch) + normalised phase (3ch) = 6ch.
    Phase is normalised to [-1, 1] range so it is on the same scale as magnitude.
    Applied after SRM filtering so both representations see the noise residual.
    """
    def __init__(self, use_fftshift=True):
        super().__init__()
        self.use_fftshift = use_fftshift

    def forward(self, x):
        # x: (B, C, H, W)
        spectrum = torch.fft.fft2(x.float())
        if self.use_fftshift:
            spectrum = torch.fft.fftshift(spectrum, dim=(-2,-1))

        # Log magnitude — same as original
        log_mag = torch.log1p(torch.abs(spectrum))
        mean = log_mag.mean(dim=(-2,-1), keepdim=True)
        std  = log_mag.std(dim=(-2,-1), keepdim=True)
        log_mag = (log_mag - mean) / (std + 1e-8)

        # Phase — normalise to [-1, 1]
        phase = torch.angle(spectrum) / torch.pi   # [-1, 1]

        # Concatenate: (B, 2C, H, W)
        return torch.cat([log_mag, phase], dim=1)


class FreqBranchP1(nn.Module):
    """Frequency branch with phase channel added."""
    def __init__(self, feature_dim=256):
        super().__init__()
        self.srm       = SRMFilterLayer()
        self.fft       = PhaseAugmentedFFT()
        # SRM gives 9ch, phase doubles it to 18ch
        self.cnn       = nn.Sequential(
            nn.Conv2d(18, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(128, feature_dim)
        self.head = nn.Linear(feature_dim, 2)

    def forward(self, images):
        patches = select_patch_v5_batch(images, patch_size=56).to(images.device)
        x = self.srm(patches)
        x = self.fft(x)
        x = self.cnn(x)
        x = self.gap(x).flatten(1)
        x = self.proj(x)
        return self.head(x)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg_p1 = make_cfg("perc_p1_phase")
model_p1 = FreqBranchP1().to(device)
acc_p1 = train_and_eval(model_p1, cfg_p1, train_loader, val_loader, test_loader, device)



Experiment: perc_p1_phase | Epochs: 30
Epoch   1/30 | train_loss=0.6710 | train_acc=58.4% | val_acc=56.7%
  -> Saved best val_acc=56.7%
Epoch   2/30 | train_loss=0.6414 | train_acc=62.5% | val_acc=59.5%
  -> Saved best val_acc=59.5%
Epoch   3/30 | train_loss=0.6286 | train_acc=63.8% | val_acc=59.2%
Epoch   4/30 | train_loss=0.6177 | train_acc=64.9% | val_acc=61.8%
  -> Saved best val_acc=61.8%
Epoch   5/30 | train_loss=0.6085 | train_acc=65.9% | val_acc=63.0%
  -> Saved best val_acc=63.0%
Epoch   6/30 | train_loss=0.5988 | train_acc=66.8% | val_acc=64.0%
  -> Saved best val_acc=64.0%
Epoch   7/30 | train_loss=0.5915 | train_acc=67.6% | val_acc=62.9%
Epoch   8/30 | train_loss=0.5858 | train_acc=68.1% | val_acc=64.9%
  -> Saved best val_acc=64.9%
Epoch   9/30 | train_loss=0.5788 | train_acc=68.6% | val_acc=66.0%
  -> Saved best val_acc=66.0%
Epoch  10/30 | train_loss=0.5754 | train_acc=68.8% | val_acc=65.9%
Epoch  11/30 | train_loss=0.5692 | train_acc=69.5% | val_acc=66.3%
  -> Saved be

## 5. P2: Multi-Scale FFT
Extract patches at three sizes (28×28, 56×56, 112×112).
Each scale fed through a shared CNN. Features concatenated before classifier.
Different generators leave artifacts at different frequency scales.


In [ ]:
class MultiScaleFreqBranch(nn.Module):
    """
    Three-scale frequency branch.
    Patches: 28x28 (fine), 56x56 (medium), 112x112 (coarse).
    Shared CNN weights across scales — forces scale-invariant feature learning.
    Features from all three scales concatenated before classifier.
    """
    def __init__(self, feature_dim=256):
        super().__init__()
        self.srm        = SRMFilterLayer()
        self.patch_sizes = [28, 56, 112]

        # Shared CNN — same weights applied to each scale
        self.shared_cnn = nn.Sequential(
            nn.Conv2d(9, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        # 3 scales × 64 features each = 192
        self.proj = nn.Linear(64 * len(self.patch_sizes), feature_dim)
        self.head = nn.Linear(feature_dim, 2)

    def forward(self, images):
        from utils.fft_utils import fft_spectrum_tensor
        scale_features = []

        for ps in self.patch_sizes:
            patches = select_patch_v5_batch(images, patch_size=ps).to(images.device)
            # Resize all patches to 56x56 so shared CNN sees same spatial size
            if ps != 56:
                patches = F.interpolate(patches, size=(56, 56), mode='bilinear',
                                        align_corners=False)
            x = self.srm(patches)
            x = fft_spectrum_tensor(x)
            x = self.shared_cnn(x)
            x = self.gap(x).flatten(1)   # (B, 64)
            scale_features.append(x)

        x = torch.cat(scale_features, dim=1)  # (B, 192)
        x = self.proj(x)
        return self.head(x)

In [7]:
cfg_p2 = make_cfg("perc_p2_multiscale")
model_p2 = MultiScaleFreqBranch().to(device)
acc_p2 = train_and_eval(model_p2, cfg_p2, train_loader, val_loader, test_loader, device)



Experiment: perc_p2_multiscale | Epochs: 30
Epoch   1/30 | train_loss=0.6580 | train_acc=61.0% | val_acc=61.2%
  -> Saved best val_acc=61.2%
Epoch   2/30 | train_loss=0.6201 | train_acc=65.9% | val_acc=56.2%
Epoch   3/30 | train_loss=0.5997 | train_acc=67.7% | val_acc=55.5%
Epoch   4/30 | train_loss=0.5838 | train_acc=69.1% | val_acc=54.1%
Epoch   5/30 | train_loss=0.5706 | train_acc=70.1% | val_acc=54.4%
Epoch   6/30 | train_loss=0.5604 | train_acc=70.8% | val_acc=61.0%
Epoch   7/30 | train_loss=0.5498 | train_acc=71.9% | val_acc=58.2%
Epoch   8/30 | train_loss=0.5410 | train_acc=72.5% | val_acc=59.5%
Epoch   9/30 | train_loss=0.5325 | train_acc=73.2% | val_acc=55.1%
Epoch  10/30 | train_loss=0.5238 | train_acc=73.8% | val_acc=56.5%
Epoch  11/30 | train_loss=0.5146 | train_acc=74.3% | val_acc=65.3%
  -> Saved best val_acc=65.3%
Epoch  12/30 | train_loss=0.5061 | train_acc=75.0% | val_acc=59.8%
Epoch  13/30 | train_loss=0.5017 | train_acc=75.2% | val_acc=56.0%
Epoch  14/30 | train_los

## 6. P3: MobileNetV3-Small Backbone
Replace FrequencyCNN with pretrained MobileNetV3-Small.
The pretrained weights give the model a better starting point for
feature extraction from structured 2D patterns like FFT spectra.
Input channels adapted from 3 to 9 (SRM output).


In [ ]:
import torchvision.models as tvm

class MobileNetFreqBranch(nn.Module):
    """
    Frequency branch with MobileNetV3-Small as the spectrum CNN.
    Pretrained on ImageNet — first conv adapted to accept 9 channels (SRM output).
    """
    def __init__(self, feature_dim=256):
        super().__init__()
        self.srm = SRMFilterLayer()

        # Load pretrained MobileNetV3-Small
        backbone = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.DEFAULT)

        # Adapt first conv from 3ch to 9ch — average pretrained weights across new channels
        orig_weight = backbone.features[0][0].weight.data  # (16, 3, 3, 3)
        new_weight  = orig_weight.repeat(1, 3, 1, 1) / 3.0  # (16, 9, 3, 3)
        backbone.features[0][0] = nn.Conv2d(9, 16, kernel_size=3, stride=2,
                                             padding=1, bias=False)
        backbone.features[0][0].weight.data = new_weight

        # Remove classifier — keep feature extractor only
        self.backbone = backbone.features
        self.gap      = nn.AdaptiveAvgPool2d(1)
        self.proj     = nn.Linear(576, feature_dim)  # MobileNetV3-Small outputs 576
        self.head     = nn.Linear(feature_dim, 2)

    def forward(self, images):
        from utils.fft_utils import fft_spectrum_tensor
        patches = select_patch_v5_batch(images, patch_size=56).to(images.device)
        x = self.srm(patches)
        x = fft_spectrum_tensor(x)
        x = self.backbone(x)
        x = self.gap(x).flatten(1)
        x = self.proj(x)
        return self.head(x)

In [9]:
cfg_p3 = make_cfg("perc_p3_mobilenet")
model_p3 = MobileNetFreqBranch().to(device)
acc_p3 = train_and_eval(model_p3, cfg_p3, train_loader, val_loader, test_loader, device)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /home/peter-kabati/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:04<00:00, 2.39MB/s]


Experiment: perc_p3_mobilenet | Epochs: 30


Epoch   1/30 | train_loss=0.6684 | train_acc=58.8% | val_acc=60.4%
  -> Saved best val_acc=60.4%
Epoch   2/30 | train_loss=0.6322 | train_acc=63.8% | val_acc=59.7%
Epoch   3/30 | train_loss=0.6233 | train_acc=64.7% | val_acc=60.6%
  -> Saved best val_acc=60.6%
Epoch   4/30 | train_loss=0.6179 | train_acc=65.0% | val_acc=62.8%
  -> Saved best val_acc=62.8%
Epoch   5/30 | train_loss=0.6100 | train_acc=65.8% | val_acc=62.8%
Epoch   6/30 | train_loss=0.6052 | train_acc=66.3% | val_acc=62.7%
Epoch   7/30 | train_loss=0.6008 | train_acc=66.7% | val_acc=62.8%
  -> Saved best val_acc=62.8%
Epoch   8/30 | train_loss=0.5977 | train_acc=67.0% | val_acc=64.4%
  -> Saved best val_acc=64.4%
Epoch   9/30 | train_loss=0.5935 | train_acc=67.4% | val_acc=64.6%
  -> Saved best val_acc=64.6%
Epoch  10/30 | train_loss=0.5899 | train_acc=67.8% | val_acc=64.6%
Epoch  11/30 | train_loss=0.5863 | train_acc=68.3% | val_acc=66.0%
  -> Saved best val_acc=66.0%
Epoch  12/30 | train_loss=0.5825 | train_acc=68.4% | 

## 7. Results

In [10]:
baseline = 0.712  # v5, no aug, 30 epochs

results_perc = {
    "Baseline (v5, FrequencyCNN)": baseline,
    "P1 — Phase channel":          acc_p1,
    "P2 — Multi-scale FFT":        acc_p2,
    "P3 — MobileNetV3 backbone":   acc_p3,
}

print("\n" + "="*60)
print("PERCEPTION EXPERIMENT — DeepDetect, 30 epochs, no aug")
print("Baseline to beat: 71.2%")
print("="*60)
for name, acc in results_perc.items():
    delta  = acc - baseline if name != "Baseline (v5, FrequencyCNN)" else 0.0
    status = "PASS ✓" if acc >= 0.70 else ("WEAK" if acc >= 0.60 else "FAIL ✗")
    marker = f"({delta:+.1%})" if delta != 0.0 else ""
    print(f"  {name:<40} {acc:.1%}  {marker:<8} {status}")
print("="*60)

best_name = max(results_perc, key=results_perc.get)
print(f"\nBest: {best_name}  ({results_perc[best_name]:.1%})")



PERCEPTION EXPERIMENT — DeepDetect, 30 epochs, no aug
Baseline to beat: 71.2%
  Baseline (v5, FrequencyCNN)              71.2%           PASS ✓
  P1 — Phase channel                       72.0%  (+0.8%)  PASS ✓
  P2 — Multi-scale FFT                     69.9%  (-1.3%)  WEAK
  P3 — MobileNetV3 backbone                68.7%  (-2.5%)  WEAK

Best: P1 — Phase channel  (72.0%)
